# Speech Models

In this notebook we deploy and test the two speech models used by the voice agent:

- **Whisper** — Speech to Text (STT) — converts audio into text
- **Higgs-Audio** — Text to Speech (TTS) — generates speech from text

Both models run on GPU (MIG slices) and are served via vLLM on OpenShift AI.

## Prerequisites

You need to be logged into OpenShift from the terminal before running these cells.

Verify you are logged in:

In [ ]:
!oc whoami
!oc project

### Test access to MaaS hosted Whisper model

The model needs a GPU (MIG 1g.18gb slice) so we have hosted this already in MaaS. Export the environment variables as follows.

In [ ]:
import subprocess, base64

model_url = "https://inference.apps.ocp.cloud.rhai-tmm.dev/voice-agents/whisper"

token_b64 = subprocess.run(
    ["oc", "get", "secret", "maas-secret", "-o",
     "jsonpath={.data.stt-token}"],
    capture_output=True, text=True
).stdout.strip()

access_token = base64.b64decode(token_b64).decode()

print(f"Model URL: {model_url}")
print(f"Token: {access_token[:20]}...")

### Test Whisper

First, let's generate a short test WAV file using Python, then send it to the Whisper API.

In [ ]:
import wave
import struct
import math

# Generate a 2-second sine wave WAV file (440 Hz tone)
sample_rate = 16000
duration = 2
frequency = 440

samples = []
for i in range(sample_rate * duration):
    t = i / sample_rate
    sample = int(32767 * 0.5 * math.sin(2 * math.pi * frequency * t))
    samples.append(struct.pack('<h', sample))

with wave.open('/tmp/test.wav', 'w') as wav:
    wav.setnchannels(1)
    wav.setsampwidth(2)
    wav.setframerate(sample_rate)
    wav.writeframes(b''.join(samples))

print('Created /tmp/test.wav (2s, 16kHz, mono)')

Now send the audio to Whisper:

In [ ]:
import subprocess, json

result = subprocess.run(
    ["curl", "-s", "-X", "POST",
     f"{model_url}/v1/audio/transcriptions",
     "-H", f"Authorization: Bearer {access_token}",
     "-F", "file=@/tmp/test.wav",
     "-F", "model=whisper"],
    capture_output=True, text=True
)

response = json.loads(result.stdout)
print(json.dumps(response, indent=2))

Since we sent a pure sine wave tone (not speech), the transcription will likely be empty or contain
artifacts. The important thing is the API responded successfully. In the voice agent pipeline,
real speech audio is sent to this endpoint.

## Text to Speech — Higgs-Audio

Higgs-Audio is the second speech model. It generates natural-sounding speech from text,
completing the voice agent loop.

We deploy it as a standard Kubernetes Deployment with a vLLM container.
It uses a GPU (MIG 2g.35gb slice) and downloads the model from Hugging Face.

### Deploy Higgs-Audio

In [ ]:
!oc apply -f ../../ai-voice-agent/deploy/models/higgs-audio/higgs-audio-v2-deployment.yaml

### Wait for Higgs-Audio to be ready

This model downloads weights from Hugging Face on first start, so it may take several minutes.
Wait until the pod shows `1/1 Running` and the readiness probe passes.

In [ ]:
!oc get pods -l app=higgs-audio-predictor
!echo '---'
!oc rollout status deployment/higgs-audio-predictor --timeout=10s 2>&1 || echo 'Still rolling out...'

### Test Higgs-Audio

Send a text prompt to the TTS model and receive audio back.
The model returns raw PCM audio (signed 16-bit little-endian, 24kHz, mono).

In [ ]:
import subprocess, json, os

# Higgs-Audio is a plain Service (not LLMInferenceService), access via cluster DNS
higgs_url = "https://higgs-audio-predictor-voice-agents.apps.ocp.cloud.rhai-tmm.dev"

result = subprocess.run(
    ["curl", "-s", "-X", "POST",
     f"{higgs_url}/v1/audio/speech",
     "-H", "Content-Type: application/json",
     "-d", json.dumps({
         "model": "higgs-audio-v2-generation-3B-base",
         "voice": "belinda",
         "input": "What would you like on your pizza?",
         "response_format": "pcm"
     }),
     "--output", "/tmp/tts-output.pcm"],
    capture_output=True, text=True
)

pcm_size = os.path.getsize('/tmp/tts-output.pcm')
print(f'Received {pcm_size} bytes of PCM audio')

Convert the raw PCM to WAV and play it in the notebook:

In [ ]:
import wave
import IPython.display as ipd

# Convert PCM (s16le, 24kHz, mono) to WAV
with open('/tmp/tts-output.pcm', 'rb') as pcm:
    pcm_data = pcm.read()

with wave.open('/tmp/tts-output.wav', 'w') as wav:
    wav.setnchannels(1)
    wav.setsampwidth(2)  # 16-bit
    wav.setframerate(24000)
    wav.writeframes(pcm_data)

print(f'Audio duration: {len(pcm_data) / (24000 * 2):.1f} seconds')
ipd.Audio('/tmp/tts-output.wav')

### Measure TTS Generation Speed (gen x)

For a real-time voice agent, TTS must generate audio faster than real-time — otherwise the user
hears silence while waiting. We measure this as **gen x**:

```
gen x = audio seconds produced / wall clock seconds elapsed
```

A gen x of **1.0** means real-time. Above 1.0 means the model generates faster than playback —
the higher the better. Below 1.0 and the user will experience latency gaps.

In [ ]:
import subprocess, json, os, time

higgs_url = "https://higgs-audio-predictor-voice-agents.apps.ocp.cloud.rhai-tmm.dev"
prompts = [
    "What would you like on your pizza?",
    "Your order has been placed and will be ready in about twenty minutes.",
    "We have pepperoni, mushrooms, olives, onions, and extra cheese available as toppings.",
]

print(f"{'Prompt':<75} {'Audio (s)':>9} {'Wall (s)':>9} {'Gen x':>7}")
print("-" * 105)

for text in prompts:
    start = time.time()
    result = subprocess.run(
        ["curl", "-s", "-X", "POST",
         f"{higgs_url}/v1/audio/speech",
         "-H", "Content-Type: application/json",
         "-d", json.dumps({
             "model": "higgs-audio-v2-generation-3B-base",
             "voice": "belinda",
             "input": text,
             "response_format": "pcm"
         }),
         "--output", "/tmp/tts-bench.pcm"],
        capture_output=True, text=True
    )
    wall_seconds = time.time() - start

    pcm_bytes = os.path.getsize('/tmp/tts-bench.pcm')
    audio_seconds = pcm_bytes / (24000 * 2)  # 24kHz, 16-bit mono
    gen_x = audio_seconds / wall_seconds if wall_seconds > 0 else 0

    print(f"{text:<75} {audio_seconds:>8.1f}s {wall_seconds:>8.1f}s {gen_x:>6.1f}x")

## Summary

You have deployed and tested both speech models:

| Model | Type | GPU | API |
|-------|------|-----|-----|
| Whisper | Speech → Text | MIG 1g.18gb | `/v1/audio/transcriptions` |
| Higgs-Audio | Text → Speech | MIG 2g.35gb | `/v1/audio/speech` |

These models form the speech layer of the voice agent pipeline:

```
User Speech → [Whisper STT] → Text → [LLM Agent] → Text → [Higgs-Audio TTS] → Speech
```

In the next section, we will deploy the LLM agent that connects these models together.